# Integration of chromatographic peaks: comparison of the methods

`peaxart.chromatogram` integrates the peaks of one chromatogram (signal vs rt, e.g. a
unit mass chromatogram) with several methods. Each returns its own peak table; they all
have `rt`, `height` and `area`.

| method | idea | coelution |
|---|---|---|
| `Trapezoid` | area above a straight baseline between bounds (half widths, prominence bases); overlapping peaks share one baseline and are split by a drop line at the valley | only peaks with their own maximum; a shoulder gets NaN |
| `Fit` | Pap-Papai peaks + constant baseline fitted (the rt stage of the extraction), extra peaks tried where the data show a shoulder | separated if the shoulder is visible |
| `FixedShapes` | positions and shapes given (e.g. fitted on a clean ion of each compound); only the heights (>= 0) and a linear baseline are fitted | split with the shapes of the compounds, works for any overlap |

Data: unit mass chromatograms of `251104.1724.air.1` (alpinac pipeline, as stored by
esper) around **2-methylhexane** (1823.2 s, only it on m/z 85) and **3-methylhexane**
(1826.1 s, only it on m/z 70 and 71), 2.9 s apart. Both compounds give m/z 41, 43, 56 and 57.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import peaxart
from peaxart.plotting import INK, MUTED, STYLE, plot_integration
from peaxart.shapes import chromato, chromato_area

plt.rcParams.update(STYLE)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.precision", 3)

z = np.load(Path("../data/examples/methylhexanes_chromatograms.npz"))
t = z["rt"]
chrom = dict(zip(z["masses"].tolist(), z["intensity"].T))
MASSES = list(chrom)
XLIM = (1812, 1840)

# a compound keeps its colour in every plot; other peaks are grey
COMPOUNDS = {"2-methylhexane": (1823.2, "#2a78d6"), "3-methylhexane": (1826.1, "#eb6834")}
RTS = [rt for rt, _ in COMPOUNDS.values()]


def compound_of(rt, tol=1.2):
    for name, (r, _) in COMPOUNDS.items():
        if abs(rt - r) < tol:
            return name
    return None


def colors(peaks):
    return [COMPOUNDS[c][1] if (c := compound_of(r)) else MUTED for r in peaks["rt"]]


def show(ax, y, peaks, method, title):
    plot_integration(ax, t, y, peaks, method, title=title, xlim=XLIM, colors=colors(peaks))

## The chromatograms

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharex=True)
for ax, m in zip(axes.flat, MASSES):
    sel = (t >= XLIM[0]) & (t <= XLIM[1])
    ax.plot(t[sel], chrom[m][sel], color=INK, lw=1)
    for name, (rt, color) in COMPOUNDS.items():
        ax.axvline(rt, color=color, lw=1, ls=":")
    ax.set_title(f"m/z {m}")
axes.flat[-1].axis("off")
axes.flat[-1].legend(handles=[plt.Line2D([], [], color=c, ls=":", label=n) for n, (_, c) in COMPOUNDS.items()],
                     loc="center", frameon=False)
fig.tight_layout()

## One shared ion, m/z 43

- **Trapezoid, detected peaks**: the slope detection (window ~3 rt sigma) sees one peak.
- **Trapezoid at the known rts**: two maxima, split by a drop line at the valley; the
  first peak also gets the tail of the second and vice versa.
- **Fit**: one peak detected, the extra-peak search adds the second (the shoulder test passes).
- **FixedShapes**: shapes fitted on m/z 85 (2-methylhexane) and m/z 70 (3-methylhexane).

In [ ]:
def window(df, lo=1820, hi=1830):
    return df[df["rt"].between(lo, hi)]


fit = peaxart.Fit()
# the shapes of the two compounds, from their clean ions
shapes = pd.concat([window(fit.integrate(t, chrom[m])) for m in (85, 70)], ignore_index=True)
fixed = peaxart.FixedShapes.from_peaks(shapes)
shapes[["rt", "rt_sigma", "rt_tailing", "area", "sn"]]

In [ ]:
y = chrom[43]
runs = {
    "trapezoid, detected": (peaxart.Trapezoid(), None),
    "trapezoid at the known rts": (peaxart.Trapezoid(), RTS),
    "fit": (fit, None),
    "fixed shapes (from m/z 85, 70)": (fixed, None),
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharey=True)
tables = {}
for ax, (title, (method, rts)) in zip(axes, runs.items()):
    tables[title] = method.integrate(t, y, rts)
    show(ax, y, tables[title], method, title)
axes[0].set_ylabel("m/z 43 (counts)")
fig.tight_layout()

Each method has its own columns:

In [ ]:
for title, df in tables.items():
    print(title)
    display(window(df))

## All ions

In [ ]:
methods = {"trapezoid at the known rts": (peaxart.Trapezoid(), RTS), "fit": (fit, None), "fixed shapes": (fixed, None)}
results = {}
fig, axes = plt.subplots(len(MASSES), 3, figsize=(13, 2.3 * len(MASSES)), sharex=True)
for row, m in zip(axes, MASSES):
    for ax, (title, (method, rts)) in zip(row, methods.items()):
        peaks = method.integrate(t, chrom[m], rts)
        results[m, title] = peaks
        show(ax, chrom[m], peaks, method, f"m/z {m}, {title}")
        ax.set_xlabel("")
for ax in axes[-1]:
    ax.set_xlabel("rt (s)")
fig.tight_layout()

### Spectra of the two compounds

Area of each compound on each ion, per method. The fit finds the second peak only where
the data show a shoulder (missing entries: the method gave no peak for that compound),
the trapezoid gives NaN for a shoulder without maximum.

In [ ]:
rows = []
for (m, title), peaks in results.items():
    for name in COMPOUNDS:
        hit = peaks[[compound_of(r) == name for r in peaks["rt"]]]
        rows.append({"compound": name, "mass": m, "method": title,
                     "area": hit["area"].iloc[0] if len(hit) else np.nan,
                     "area_u": hit["area_u"].iloc[0] if len(hit) and "area_u" in hit else np.nan})
areas = pd.DataFrame(rows)
areas.pivot_table(index="mass", columns=["compound", "method"], values="area", dropna=False).round(-3)

In [ ]:
areas.pivot_table(index="mass", columns=["compound", "method"], values="area_u", dropna=False)

In [ ]:
# relative spectra (% of the largest ion of the compound), one panel per compound
MARKERS = {"trapezoid at the known rts": "s", "fit": "o", "fixed shapes": "D"}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, (name, (_, color)) in zip(axes, COMPOUNDS.items()):
    sub = areas[areas["compound"] == name]
    for k, (title, marker) in enumerate(MARKERS.items()):
        s = sub[sub["method"] == title].set_index("mass")["area"]
        x = np.arange(len(MASSES)) + (k - 1) * 0.22
        ax.vlines(x, 0, 100 * s / s.max(), color=color, lw=2, alpha=0.35 + 0.3 * k)
        ax.plot(x, 100 * s / s.max(), marker, color=color, ms=6, alpha=0.35 + 0.3 * k, label=title)
    ax.set_xticks(range(len(MASSES)), [str(m) for m in MASSES])
    ax.set_xlabel("m/z")
    ax.set_title(name)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("relative area (%)")
fig.tight_layout()

## Time

Time of `integrate` on each ion (50 s, 300 points; best of 5 runs). `FixedShapes` does
not include the fits of the shapes on m/z 85 and 70 (`shapes`, done once per compound pair).
`fit, legacy numerics` is the `Fit` with `ExtractionConfig(legacy_numerics=True)` (the
bit-for-bit reproduction of the legacy code) for comparison.

In [ ]:
import time


def best_time(f, repeat=5):
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        f()
        times.append(time.perf_counter() - start)
    return min(times)


timed = {
    "trapezoid, detected": (peaxart.Trapezoid(), None),
    "trapezoid at the known rts": (peaxart.Trapezoid(), RTS),
    "fit, detected": (fit, None),
    "fit at the known rts": (fit, RTS),
    "fit, legacy numerics": (peaxart.Fit(peaxart.ExtractionConfig(convention="standard")), None),
    "fixed shapes": (fixed, None),
}
times = pd.DataFrame({
    name: {m: 1e3 * best_time(lambda: method.integrate(t, chrom[m], rts)) for m in MASSES}
    for name, (method, rts) in timed.items()
}).rename_axis("m/z")
shape_time = 1e3 * best_time(lambda: [fit.integrate(t, chrom[m]) for m in (85, 70)])
print(f"shapes (fits of m/z 85 and 70): {shape_time:.0f} ms")
times.round(1).T.assign(mean=times.mean().round(1)).rename_axis("time (ms)")

For a whole run: a synthetic chromatogram of 3000 s (18 000 points, the length of a run)
with 100 peaks of random heights and positions, some coeluting. The time of a run of
esper is about this times the number of unit masses (~300).

In [ ]:
rng = np.random.default_rng(2)
RUN_SIGMA, RUN_TAILING = 1.4, 0.3
t_run = np.arange(0, 3000, 1 / 6)
run_peaks = np.column_stack([10 ** rng.uniform(3, 6, 100), np.sort(rng.uniform(50, 2950, 100))])
y_run = 500 + sum(chromato(t_run, h, r, RUN_SIGMA, RUN_TAILING) for h, r in run_peaks)
y_run = y_run + rng.normal(0, 50, len(t_run))
run_methods = {
    "trapezoid, detected": (peaxart.Trapezoid(), None),
    "fit, detected": (fit, None),
    "fit at the known rts": (fit, run_peaks[:, 1]),
    "fixed shapes (true shapes)": (peaxart.FixedShapes(run_peaks[:, 1], RUN_SIGMA, RUN_TAILING), None),
}
rows = []
for name, (method, rts) in run_methods.items():
    seconds = best_time(lambda: method.integrate(t_run, y_run, rts), repeat=2)
    n = len(method.integrate(t_run, y_run, rts))
    rows.append({"method": name, "time (s)": seconds, "peaks": n, "300 masses (min)": 300 * seconds / 60})
pd.DataFrame(rows).set_index("method").round(2)

## Synthetic coelution: error of the areas

Two Pap-Papai peaks (heights 1 and 0.8, sigma 1.4 s, tailing 0.3), noise 0.5 % of the
first height, at increasing separation. Error of the area of each peak, per method;
`FixedShapes` gets the true shapes (as from clean ions), the trapezoid the true rts.
`trapezoid, no drop line` gives each peak its own baseline between its own bounds (from the valley).

In [ ]:
SIGMA, TAILING, H = 1.4, 0.3, (1e5, 8e4)
true_area = [chromato_area(h, SIGMA, TAILING) for h in H]
ts = np.arange(1800, 1850, 1 / 6)
bench_methods = {
    "trapezoid, no drop line": lambda y, rts: peaxart.Trapezoid(split_at_valley=False).integrate(ts, y, rts),
    "trapezoid": lambda y, rts: peaxart.Trapezoid().integrate(ts, y, rts),
    "fit": lambda y, rts: fit.integrate(ts, y),
    "fixed shapes": lambda y, rts: peaxart.FixedShapes(rts, SIGMA, TAILING).integrate(ts, y),
}
rows = []
rng = np.random.default_rng(1)
for d in np.arange(1.0, 5.01, 0.25):  # separation in sigma
    rts = [1820.0, 1820.0 + d * SIGMA]
    for rep in range(5):
        y = 500 + sum(chromato(ts, h, r, SIGMA, TAILING) for h, r in zip(H, rts))
        y = y + rng.normal(0, 0.005 * H[0], len(ts))
        for name, run in bench_methods.items():
            peaks = run(y, rts)
            for k, r in enumerate(rts):
                hit = peaks[(peaks["rt"] - r).abs() < 0.5 * SIGMA] if len(peaks) else peaks
                a = hit["area"].iloc[0] if len(hit) else np.nan
                rows.append({"separation": d, "method": name, "peak": k, "error": 100 * (a / true_area[k] - 1)})
bench = pd.DataFrame(rows)

In [ ]:
BENCH_STYLE = {"trapezoid, no drop line": ("#e87ba4", ":"), "trapezoid": ("#eda100", "-."),
               "fit": ("#1baf7a", "--"), "fixed shapes": ("#2a78d6", "-")}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, k in zip(axes, (0, 1)):
    sub = bench[bench["peak"] == k]
    for name, (color, ls) in BENCH_STYLE.items():
        g = sub[sub["method"] == name].groupby("separation")["error"]
        found = g.apply(lambda e: e.notna().mean())
        med = g.median()
        ax.plot(med.index, med.values, ls, color=color, lw=2, label=name)
        # separations where the method gives no area in some repetitions: open markers
        partial = found[found < 1].index
        ax.plot(partial, med.reindex(partial), "o", mfc="white", color=color, ms=6)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_ylim(-60, 60)
    ax.set_xlabel("separation (rt sigma)")
    ax.set_title(f"peak {k + 1} (height {H[k] / H[0]:.1f})")
axes[0].set_ylabel("area error, median of 5 (%)")
axes[1].legend(frameon=False)
fig.tight_layout()

In [ ]:
# share of the repetitions where each method gives an area for the second peak
(bench[bench["peak"] == 1].assign(found=lambda d: d["error"].notna())
 .pivot_table(index="separation", columns="method", values="found").T.round(1))

### Sensitivity of `FixedShapes` to the shapes

The benchmark above gives `FixedShapes` the exact shapes. With shapes from other ions they
are slightly off: here 2 sigma apart, error of the two areas (%) for a shift of both rts
and a sigma too wide.

In [ ]:
rts = [1820.0, 1820.0 + 2 * SIGMA]
y = 500 + sum(chromato(ts, h, r, SIGMA, TAILING) for h, r in zip(H, rts))
y = y + np.random.default_rng(0).normal(0, 0.005 * H[0], len(ts))
rows = []
for shift in (0.0, 0.05, 0.1, 0.2):
    for scale in (1.0, 1.1):
        peaks = peaxart.FixedShapes([r + shift for r in rts], SIGMA * scale, TAILING).integrate(ts, y)
        err = 100 * (peaks["area"].to_numpy() / true_area - 1)
        rows.append({"rt shift (s)": shift, "sigma x": scale, "peak 1 (%)": err[0], "peak 2 (%)": err[1]})
pd.DataFrame(rows).round({"peak 1 (%)": 1, "peak 2 (%)": 1})

## Summary

- **Trapezoid** needs a maximum per peak. On the methylhexanes it separates the two
  compounds only where both have a top (m/z 41, 43, 57), and the drop line gives the tail of
  2-methylhexane to 3-methylhexane (m/z 43: 2.3e6 / 4.9e6 against 3.8e6 / 3.3e6 with the
  shapes). A shoulder gets NaN and its area goes to the main peak (m/z 56, 70, 71, 85).
  For isolated peaks it is fine and needs no model. Without the drop line, peaks start
  from the valley and lose the signal under it (large errors in the benchmark).
- **Fit** separates the two compounds wherever the shoulder is visible (above ~2 rt sigma,
  and the second peak at least 10 % of the first: on m/z 71 the small 2-methylhexane
  shoulder is left in the residual). Within ~5 % in the benchmark, but the shapes are
  free on each ion, so the split of the overlap varies from ion to ion (2-methylhexane gets
  2.2e6 on m/z 41 by the fit, 1.9e6 with the shapes). It also gives uncertainties, LOD and S/N.
- **FixedShapes** splits any overlap, even without a shoulder, with the same shapes on every
  ion; a compound absent from an ion gets 0 (m/z 70 and 85). It needs the shapes from clean
  ions or from a fit, and is sensitive to them: an rt off by 0.1 s changes the areas by up to
  4 %, a sigma 10 % too wide by up to 10 %. Possible improvement: let the rts move slightly (nonlinear
  refinement with the widths fixed).

For esper: `Fit` for the areas of every peak, and `FixedShapes` to update the areas of a
compound's ions when compounds coelute (shapes from the ions where each compound is alone).